In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
def cart_position_category(add_to_cart_order):
    if add_to_cart_order == 1:
        return "First Pick"
    elif 2 <= add_to_cart_order <= 5:
        return "Essential"
    else:
        return "Add-on"

In [0]:
cart_position_udf = F.udf(cart_position_category, StringType())

In [0]:
df_order_products = spark.table(f"{catalogo}.{esquema_source}.order_products__train")
df_products = spark.table(f"{catalogo}.{esquema_source}.products")

In [0]:
df_order_products = df_order_products.dropna(how="all") \
    .filter((col("order_id").isNotNull()) | (col("product_id").isNotNull()))

df_products = df_products.dropna(how="all") \
    .filter((col("product_id").isNotNull()) | (col("product_name").isNotNull()))

In [0]:
df_order_products = df_order_products.withColumn("cart_position_category", cart_position_udf("add_to_cart_order"))

In [0]:
df_joined = df_order_products.alias("x").join(df_products.alias("y"), col("x.product_id") == col("y.product_id"), "inner")

In [0]:
df_filtered_sorted = df_joined.filter(col("x.order_id") > 0).orderBy("x.order_id")

In [0]:
df_transformed = df_filtered_sorted.withColumn(
    "reorder_desc",
    F.when(F.col("x.reordered") == 1, F.lit("Repeat Purchase")).otherwise(F.lit("New Purchase"))
)

In [0]:
df_aggregated = df_transformed.groupBy("y.department_id").agg(
    F.count("x.order_id").alias("temp_count")
)

In [0]:
df_with_organic = df_transformed.withColumn(
    "is_organic",
    F.when(F.col("y.product_name").contains("Organic"), F.lit("Y")).otherwise(F.lit("N"))
)

In [0]:
df_updated = df_with_organic.select(
    col("x.order_id"),
    col("x.product_id"),
    col("y.product_name"),
    col("y.aisle_id"),
    col("y.department_id"),
    col("x.add_to_cart_order"),
    col("x.cart_position_category"),
    col("x.reordered"),
    col("reorder_desc"),
    col("is_organic")
).withColumn("ingestion_date", F.current_timestamp())

In [0]:
df_updated.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.products_orders_enriched")